# Model odrađenosti bloka — backtest (Etapa 11a)

Bayesovska logistička regresija (`src/lib/slots/completion.ts`, isti u `hoot_ml/completion.py`) sa predmetom kao hijerarhijskim efektom. Vremenski backtest: blokovi svakog dana se predviđaju iz blokova ranijih dana. Metrike: Brier i log-loss, model sa predmetom prema modelu bez predmeta i prema konstanti (udio odrađenih do tada). Ishod: odrađen = 1, sve ostalo 0.

Izvor podataka redom: izvoz (`ml/data/*.parquet`), primjeri simulatora (`npm run sim -- --seeds 30 --examples ml/data/sim_blocks.json`), inače sintetički primjeri.

In [ ]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from hoot_ml.completion import fit, subject_effects
from hoot_ml.completion_fit import (
    MIN_FIT_BLOCKS,
    backtest,
    examples_from_frame,
    fit_tau,
    sim_examples,
    synthetic_examples,
)
from hoot_ml.data import block_examples, load

data = Path("../data")
today = date.today().isoformat()
if any(data.glob("*.parquet")):
    source = "stvarni podaci"
    runs = {"korisnik": examples_from_frame(block_examples(load(data), today))}
elif (data / "sim_blocks.json").exists():
    source = "simulator"
    runs = {f"run {r}": sim_examples(data / "sim_blocks.json", r) for r in range(1, 31)}
else:
    source = "sintetički podaci"
    runs = {f"seed {s}": synthetic_examples(s) for s in range(10)}
print(source, sum(len(x) for x in runs.values()), "blokova")

## Backtest: sa predmetom, bez predmeta, konstanta

In [ ]:
results = pd.DataFrame({name: vars(backtest(x)) for name, x in runs.items()}).T
results.mean().round(4)

In [ ]:
# Razlika po runu (negativno = model sa predmetom bolji) i njena standardna greška.
diff = results["log_loss"] - results["log_loss_no_subject"]
pd.Series({"diff": diff.mean(), "se": diff.std() / np.sqrt(len(diff))}).round(4)

## Efekat predmeta (ulaz uvida „Izbjegavanje“)

Uslovni sd (s obzirom na fiksne efekte). Značajno izbjegavanje: z < −1,645.

In [ ]:
last, examples = next(reversed(runs.items()))
end = max(e.on_date for e in examples)
effects = pd.DataFrame(subject_effects(fit(examples, end)))
effects["z"] = effects["mean"] / effects["sd"]
effects.round(3)

## Fit priora

S jednim korisnikom se fituje samo τ_predmet (Laplaceova evidencija); srednje vrijednosti koeficijenata ostaju dok nema više korisnika (inače bi se isti blokovi brojali dvaput). Novi `priors.json` (nova `version` → `input_hash`) tek od dovoljno blokova.

In [ ]:
blocks = sum(len(x) for x in runs.values())
if source == "stvarni podaci" and blocks >= MIN_FIT_BLOCKS:
    print("tauSubject", fit_tau(runs["korisnik"], today))
else:
    print(f"{source}, {blocks} blokova: prior ostaje")